# 1. Import libraries

In [9]:
import pandas as pd
import numpy as np
import os
import time
from joblib import dump

from sklearn.model_selection import StratifiedKFold, cross_val_predict, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, make_scorer, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# 2. Load data

In [2]:
data = pd.read_excel('złączone_dane.xlsx')
data = data.drop('image_id',axis=1)
data = data.drop(columns=[col for col in data.columns if any(x in col for x in ['3_p', '4_p', '5_p'])])

# 3. Preprocessing

In [10]:
X = data.drop('label', axis=1)
y = data['label']
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Definicja funkcji PCA na grupach
def apply_grouped_pca(X, n_components=10):
    lm_0_cols = [col for col in X.columns if col.startswith('0_point_lm_')]
    lm_1_cols = [col for col in X.columns if col.startswith('1_point_lm_')]
    lm_2_cols = [col for col in X.columns if col.startswith('2_point_lm_')]
    vec_cols = [col for col in X.columns if '_vec_' in col]

    def pca_transform(cols, prefix):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X[cols])
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_scaled)
        return pd.DataFrame(X_pca, columns=[f'{prefix}_pca_{i}' for i in range(n_components)], index=X.index)

    pca_lm_0 = pca_transform(lm_0_cols, '0')
    pca_lm_1 = pca_transform(lm_1_cols, '1')
    pca_lm_2 = pca_transform(lm_2_cols, '2')
    vec_features = X[vec_cols].reset_index(drop=True)

    X_pca = pd.concat([pca_lm_0, pca_lm_1, pca_lm_2, vec_features], axis=1)
    return X_pca

X_pca = apply_grouped_pca(X, n_components=10)

# 4. Ustawienia

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro'),
    'recall': make_scorer(recall_score, average='macro'),
    'f1': make_scorer(f1_score, average='macro')
}

os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)
os.makedirs('reports', exist_ok=True)


# 5. Pipeline

In [12]:
pipeline = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 6. Parametry do losowego przeszukiwania

In [13]:
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [5, 10, 20, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2']
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=20,
    scoring='f1_macro',
    n_jobs=-1,
    cv=cv,
    verbose=1,
    random_state=42
)

# 7. Trening + dopasowanie

In [ ]:
print(f'\n🔍 Start treningu RandomForest z losowym doborem hiperparametrów...')
start_time = time.time()
search.fit(X_pca, y_encoded)
training_time = time.time() - start_time

best_model = search.best_estimator_


🔍 Start treningu RandomForest z losowym doborem hiperparametrów...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


# 8. Predykcja i ocena

In [ ]:
y_pred = cross_val_predict(best_model, X_pca, y_encoded, cv=cv)
report = classification_report(y_encoded, y_pred, digits=4)

# Średnie metryki
results = cross_val_predict(best_model, X_pca, y_encoded, cv=cv)
scores = cross_val_predict(best_model, X_pca, y_encoded, cv=cv, method='predict_proba')

# Zapisywanie modelu
dump(best_model, 'models/Randomforest_best.pkl')

# Zapisywanie raportu
avg_scores = cross_val_predict(best_model, X_pca, y_encoded, cv=cv)
report_text = classification_report(y_encoded, avg_scores, digits=4)

with open('reports/Randomforest_report.txt', 'w', encoding='utf-8') as f:
    f.write(f"Najlepszy model: RandomForest\n")
    f.write(f"Parametry: {search.best_params_}\n\n")
    f.write("=== Raport klasyfikacji ===\n")
    f.write(report_text)
    f.write(f"\nCzas treningu: {training_time:.2f} sekund\n")

# Log
with open('logs/Randomforest_log.txt', 'w', encoding='utf-8') as f:
    f.write(f"Najlepszy model: RandomForest\n")
    f.write(f"Parametry: {search.best_params_}\n")
    f.write(f"Czas treningu: {training_time:.2f} sekund\n")

# Konsola
print('|====================|')
print("📄 Raport:\n", report_text)
print(f"Czas treningu: {training_time:.2f} s")
print('|====================|')

# 9. Przykład użycia

In [ ]:
sample = [X_pca.iloc[0]]
proba = best_model.predict_proba(sample)[0]
class_names = le.inverse_transform(np.arange(len(proba)))
proba_dict = dict(zip(class_names, np.round(proba, 4)))
print('\n📊 Prawdopodobieństwa dla pierwszej próbki:')
print(proba_dict)